# `02-data-cleaning-and-exploration.ipynb`: Cleaning the scraped Arrests and Arrests & Incidents Logs and doing some preliminary data exploration

**Written by**: Diego Torrealba

**Project**: umpd-crime

**Date**: 2026-11-07

## Setup

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import shutil
from pathlib import Path
from umpd_crime.constants import TOD_CATEGORIES, SEMESTER_RANGES, SEMESTER_LABELS, MONTHS

BASE_DIR = Path.cwd().parent
RAW_DATA_DIR = BASE_DIR / "data" / "raw" 
CURRENT_YEAR = time.localtime().tm_year
CURRENT_DAY = pd.to_datetime(f'{CURRENT_YEAR}-{time.localtime().tm_mon}-{time.localtime().tm_mday}')


def get_semester_conditions(series):
  """Returns list of Boolean conditions to set semester time frames, written by GPT-6.1 Sol on 2026-11-06 (stumped on dynamic condition creation)"""
  
  dates = pd.to_datetime(series).dt.normalize()
  
  return [
    dates.between(
      semester['start_date'],
      semester['end_date'],
      inclusive="both"
    )
    for semester in SEMESTER_RANGES.values()
  ]
  
def get_semester_season_conditions(series):
  
  series = pd.Series(series)
  
  return [series.str.startswith("Fall"),
          series.str.startswith("Winter"),
          series.str.startswith("Spring"),
          series.str.startswith("Summer")]

arrests_df = pd.read_csv(RAW_DATA_DIR / "arrest-ledger" / f"umpd_arrest_ledger_2018-{CURRENT_YEAR}.csv")
dci_df = pd.read_csv(RAW_DATA_DIR / 'daily-crime-incident-logs' / f'daily_crime_incident_log_2019-{CURRENT_YEAR}.csv')

In [2]:
CURRENT_DAY

Timestamp('2026-11-11 00:00:00')

## Arrests Log

In [3]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3731 entries, 0 to 3730
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   arrest_number     3731 non-null   int64  
 1   arrest_date       3731 non-null   str    
 2   umpd_case_number  3731 non-null   str    
 3   age               27 non-null     float64
 4   race              3653 non-null   str    
 5   sex               3654 non-null   str    
 6   arrested_charge   3731 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 204.2 KB


The columns that have values for every row are: `arrest_number`, `arrest_date`, `umpd_case_number` and `arrested_charge`. 

The columns that have missing values by order of how many they're missing: 

In [4]:
arrests_df.isna().sum()

arrest_number          0
arrest_date            0
umpd_case_number       0
age                 3704
race                  78
sex                   77
arrested_charge        0
dtype: int64

This checks out, as `age` is absent from most rows on the website. 

### Fixing datatypes

In [5]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3731 entries, 0 to 3730
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   arrest_number     3731 non-null   int64  
 1   arrest_date       3731 non-null   str    
 2   umpd_case_number  3731 non-null   str    
 3   age               27 non-null     float64
 4   race              3653 non-null   str    
 5   sex               3654 non-null   str    
 6   arrested_charge   3731 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 204.2 KB


- `arrest_number` should be a string, as its an identifier;
- `arrest_date` should be a datetime object

In [6]:
# Converting arrests_df columns to appropriate data types
arrests_df['arrest_number'] = arrests_df['arrest_number'].astype(str)
arrests_df['arrest_date'] = pd.to_datetime(arrests_df['arrest_date'])

### Creating supplementary columns from given columns

In [7]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3731 entries, 0 to 3730
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   arrest_number     3731 non-null   str           
 1   arrest_date       3731 non-null   datetime64[us]
 2   umpd_case_number  3731 non-null   str           
 3   age               27 non-null     float64       
 4   race              3653 non-null   str           
 5   sex               3654 non-null   str           
 6   arrested_charge   3731 non-null   str           
dtypes: datetime64[us](1), float64(1), str(5)
memory usage: 204.2 KB


In [8]:
arrests_df['time_of_day'] = pd.cut(
  arrests_df['arrest_date'].dt.hour,
  bins=TOD_CATEGORIES['bins'],
  labels=TOD_CATEGORIES['labels'],
  ordered=False,
  include_lowest=True
)
arrests_df[['arrest_date', 'time_of_day']].sample(10)

,arrest_date,time_of_day
3086,2025-08-30 01:15:00,night
998,2019-11-12 00:00:00,night
1350,2020-11-15 03:43:00,night
1997,2022-09-13 23:09:00,evening
2485,2024-03-30 12:00:00,afternoon
3106,2025-09-13 02:27:00,night
2871,2025-02-07 00:34:00,night
1152,2020-03-01 02:21:00,night
1755,2022-02-09 08:57:00,morning
1918,2022-07-23 00:00:00,night


In [9]:
arrests_df['time_of_day'].value_counts(sort=True)

time_of_day
night        1964
evening       761
afternoon     734
morning       272
Name: count, dtype: int64

In [10]:
arrests_df[arrests_df['time_of_day'] == 'night']['arrest_date'].sample(10)

2871   2025-02-07 00:34:00
1552   2021-07-08 00:52:00
1569   2021-07-25 01:28:00
2519   2024-04-22 00:00:00
2712   2024-10-11 00:53:00
2852   2025-01-16 02:51:00
483    2019-03-01 02:11:00
1743   2022-01-23 01:36:00
3678   2026-09-25 02:24:00
516    2019-03-17 01:33:00
Name: arrest_date, dtype: datetime64[us]

In [11]:
arrests_df_sem_conditions = get_semester_conditions(arrests_df['arrest_date'])
arrests_df['arrest_date_semester'] = np.select(arrests_df_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")
arrests_df_sem_season_conditions = get_semester_season_conditions(arrests_df['arrest_date_semester'])
arrests_df['arrest_date_semester_season'] = np.select(
  arrests_df_sem_season_conditions,
  ['Fall', 'Winter', 'Spring', "Summer"],
  default = "Not during a semester"
)
arrests_df['arrest_date_semester_season'].value_counts()

arrest_date_semester_season
Spring                   1299
Fall                     1169
Summer                    755
Not during a semester     354
Winter                    154
Name: count, dtype: int64

### Splitting charges

In [12]:
arrests_df['arrested_charges'] = arrests_df['arrested_charge'].str.split(';')
arrests_df['arrested_charges_num'] = arrests_df['arrested_charges'].apply(lambda x: len(x))
arrests_df = arrests_df[['arrest_number', 'umpd_case_number', 'arrest_date', 'arrest_date_semester', 'arrest_date_semester_season', 'age', 'race', 'sex', 'arrested_charge', 'arrested_charges_num', 'arrested_charges']]
arrests_df_exploded = arrests_df.explode('arrested_charges')
arrests_df_exploded['arrested_charges'] = arrests_df_exploded['arrested_charges'].str.strip()

In [13]:
arrests_df_exploded['arrested_charges_num'].describe()

count    8724.000000
mean        4.088721
std         2.812073
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max        15.000000
Name: arrested_charges_num, dtype: float64

In [14]:
arrests_df_exploded[arrests_df_exploded['arrested_charges_num'] == 15]

,arrest_number,umpd_case_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,"(Driving, Attempting to drive) veh. while impa..."
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,"(Driving, Attempting to drive) veh. while unde..."
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,"(Driving, Attempting to drive) veh. while unde..."
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver changing lanes when unsafe
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver failure to obey properly placed traffic...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver failure to sound horn warning to pedest...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver failure to stop for pedestrian in cross...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driving veh. in excess of reasonable and prude...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driving veh. on hwy. at speed exceeding speed ...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Exceeding highway work zone speed limit


In [15]:
arrests_df_exploded['arrested_charges'].value_counts(sort=True).head(15)

arrested_charges
(Driving, Attempting to drive) veh. while under the influence of alcohol                                             1284
(Driving, Attempting to drive) veh. while impaired by alcohol                                                        1175
(Driving, Attempting to drive) veh. while under the influence of alcohol per se                                       732
Reckless driving veh. in wanton and willfull disregard for safety of persons and property                             489
CDS:Possession Marijuana L/T 10 grams                                                                                 457
Negligent driving veh. in careless and imprudent manner endangering property, life and person                         457
School Trespassing On Ground                                                                                          165
THEFT: $100 TO UNDER $1,500-All Other                                                                                 153
Driver 

In [16]:
len(arrests_df_exploded['arrested_charges'].unique().tolist())

442

I gave GPT-6 Astra Light the list of 442 unique charges and asked it to categorize them.

In [17]:
arrests_df_gpt_categories = pd.read_csv(RAW_DATA_DIR.parent / 'processed' / 'gpt_charge_categories.csv')
arrests_df_exploded = pd.merge(arrests_df_exploded, arrests_df_gpt_categories, 'left', left_on='arrested_charges', right_on="charge_label")
arrests_df_exploded = arrests_df_exploded.rename({'category': 'gpt_category'}, axis=1)

I also gave Claude Sonnet 5.5 High the list of charges to see how the categories between the two models differed.

In [18]:
arrests_df_exploded

,arrest_number,umpd_case_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges,source_position,charge_label,gpt_category
0,21483,2018-00000017,2018-01-01 02:15:00,Not during a semester,Not during a semester,NaN,NaN,NaN,"(Driving, Attempting to drive) veh. while impa...",2,"(Driving, Attempting to drive) veh. while impa...",1,"(Driving, Attempting to drive) veh. while impa...",Impaired driving and related driving restrictions
1,21483,2018-00000017,2018-01-01 02:15:00,Not during a semester,Not during a semester,NaN,NaN,NaN,"(Driving, Attempting to drive) veh. while impa...",2,"(Driving, Attempting to drive) veh. while unde...",2,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions
2,21484,2017-00078928,2018-01-03 14:35:00,Winter 2018,Winter,20.0,Black,Male,"THEFT: $100 TO UNDER $1,500",1,"THEFT: $100 TO UNDER $1,500",3,"THEFT: $100 TO UNDER $1,500","Theft, theft schemes and unauthorized use or r..."
3,21485,2018-00001737,2018-01-09 12:02:00,Winter 2018,Winter,NaN,White,Male,Assault-1st Degree; Other Dangerous Weapon; Da...,3,Assault-1st Degree,4,Assault-1st Degree,"Assault, endangerment and unlawful restraint"
4,21485,2018-00001737,2018-01-09 12:02:00,Winter 2018,Winter,NaN,White,Male,Assault-1st Degree; Other Dangerous Weapon; Da...,3,Other Dangerous Weapon,5,Other Dangerous Weapon,Source descriptors and incomplete labels
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8719,25190,2026-00295024,2026-11-07 02:03:00,Fall 2026,Fall,NaN,Black,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while unde...",2,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions
8720,25190,2026-00295024,2026-11-07 02:03:00,Fall 2026,Fall,NaN,Black,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while unde...",16,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions
8721,25191,2026-00296753,2026-11-09 02:22:00,Fall 2026,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while impa...",1,"(Driving, Attempting to drive) veh. while impa...",Impaired driving and related driving restrictions
8722,25191,2026-00296753,2026-11-09 02:22:00,Fall 2026,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while unde...",2,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions


In [19]:
arrests_df_exploded = arrests_df_exploded[['arrest_number', 'umpd_case_number', 'arrest_date', 'arrest_date_semester', 'arrest_date_semester_season', 'age', 'race', 'sex', 'arrested_charge', 'arrested_charges_num', 'arrested_charges', 'gpt_category']]
arrests_df_exploded['gpt_category'].value_counts()

gpt_category
Impaired driving and related driving restrictions                                   3397
Moving violations and road-user safety                                              1483
Theft, theft schemes and unauthorized use or removal of property                     780
Drug possession, use and paraphernalia (including cannabis)                          684
Licensing, registration and insurance                                                275
Trespassing and unlawful entry                                                       260
Property damage, burning and tampering                                               211
Burglary, vehicle break-ins and burglary tools                                       197
Alcohol possession, consumption, furnishing and public intoxication                  179
Assault, endangerment and unlawful restraint                                         170
Fraud, forgery, false identification and financial offenses                          165
Source d

## Daily Crime Incidents and Arrests

In [20]:
dci_df.head()

,umpd_case_number,date_occurred,report_date,crime_type,disposition,location
0,2019-00000001,01/01/2019 00:01:00,01/01/2019 - 00:01,Fireworks Complaint,Arrest,4300 block of Knox Rd
1,2019-00000009,01/01/2019 01:20:00,01/01/2019 - 01:20,DWI/DUI,Arrest,8300 block of Baltimore Ave
2,2019-00000011,01/01/2019 01:28:00,01/01/2019 - 01:28,DWI/DUI,Arrest,7400 block of Baltimore Ave
3,2019-00000203,12/28/2018 17:00:00,01/02/2019 - 11:34,Vandalism,CBE,7500 block of Calvert Service Ln
4,2019-00000312,01/02/2019 23:04:00,01/02/2019 - 23:04,CDS Violation,Arrest,Metzerott Rd


In [21]:
dci_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12918 entries, 0 to 12917
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   umpd_case_number  12918 non-null  str  
 1   date_occurred     12918 non-null  str  
 2   report_date       12918 non-null  str  
 3   crime_type        12918 non-null  str  
 4   disposition       12918 non-null  str  
 5   location          12872 non-null  str  
dtypes: str(6)
memory usage: 605.7 KB


- `date_occurred` and `report_date` should be datetime objects.

In [22]:
# Around 51 out of over 12000 have no time stamps. Setting them to midnight
dci_df.loc[dci_df['date_occurred'].str.contains(":") != True, 'date_occurred'] = dci_df.loc[dci_df['date_occurred'].str.contains(":") != True, 'date_occurred'] + " 00:00:00"

dci_df['date_occurred'] = pd.to_datetime(dci_df['date_occurred'])
dci_df['report_date'] = pd.to_datetime(dci_df['report_date'])

In [23]:
dci_df.head()

,umpd_case_number,date_occurred,report_date,crime_type,disposition,location
0,2019-00000001,2019-01-01 00:01:00,2019-01-01 00:01:00,Fireworks Complaint,Arrest,4300 block of Knox Rd
1,2019-00000009,2019-01-01 01:20:00,2019-01-01 01:20:00,DWI/DUI,Arrest,8300 block of Baltimore Ave
2,2019-00000011,2019-01-01 01:28:00,2019-01-01 01:28:00,DWI/DUI,Arrest,7400 block of Baltimore Ave
3,2019-00000203,2018-12-28 17:00:00,2019-01-02 11:34:00,Vandalism,CBE,7500 block of Calvert Service Ln
4,2019-00000312,2019-01-02 23:04:00,2019-01-02 23:04:00,CDS Violation,Arrest,Metzerott Rd


In [24]:
dci_df['date_occurred_tod'] = pd.cut(
  dci_df['date_occurred'].dt.hour,
  bins = TOD_CATEGORIES['bins'],
  labels = TOD_CATEGORIES['labels'],
  ordered = False,
  include_lowest = True
)

dci_df_date_occurred_sem_conditions = get_semester_conditions(dci_df['date_occurred'])
dci_df['date_occurred_sem'] = np.select(dci_df_date_occurred_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")

dci_df_date_occurred_sem_season_conditions = get_semester_season_conditions(dci_df['date_occurred_sem'])

dci_df['date_occurred_sem_season'] = np.select(
  dci_df_date_occurred_sem_season_conditions,
  ['Fall', "Winter", "Spring", "Summer"],
  default = "Not during a semester"
)

In [25]:
dci_df['report_date_tod'] = pd.cut(
  dci_df['report_date'].dt.hour,
  bins = TOD_CATEGORIES['bins'],
  labels = TOD_CATEGORIES['labels'],
  ordered = False,
  include_lowest = True
)
dci_df_report_date_sem_conditions = get_semester_conditions(dci_df['report_date'])
dci_df['report_date_sem'] = np.select(dci_df_report_date_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")

dci_df_report_date_sem_season_conditions = get_semester_season_conditions(dci_df['report_date_sem'])

dci_df['report_date_sem_season'] = np.select(
  dci_df_date_occurred_sem_season_conditions,
  ['Fall', 'Winter', 'Spring', 'Summer'],
  default= 'Not during a semester'
)

In [26]:
dci_df = dci_df[['umpd_case_number', 'date_occurred', 'date_occurred_tod', 'date_occurred_sem', 'date_occurred_sem_season', 'report_date', 'report_date_tod', 'report_date_sem', 'report_date_sem_season', 'crime_type', 'disposition', 'location']]

In [27]:
dci_df

,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,crime_type,disposition,location
0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,Fireworks Complaint,Arrest,4300 block of Knox Rd
1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,DWI/DUI,Arrest,8300 block of Baltimore Ave
2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,DWI/DUI,Arrest,7400 block of Baltimore Ave
3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,Vandalism,CBE,7500 block of Calvert Service Ln
4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,CDS Violation,Arrest,Metzerott Rd
...,...,...,...,...,...,...,...,...,...,...,...,...
12913,2026-00289157,2026-10-31 23:56:00,evening,Fall 2026,Fall,2026-10-31 23:56:00,evening,Fall 2026,Fall,DWI/DUI,Arrest,Baltimore Ave
12914,2026-00289161,2026-11-01 00:00:00,night,Fall 2026,Fall,2026-11-01 00:00:00,night,Fall 2026,Fall,Injured/Sick Person,CBE,3900 block of Denton Service Ln
12915,2026-00289166,2026-11-01 00:08:00,night,Fall 2026,Fall,2026-11-01 00:08:00,night,Fall 2026,Fall,Injured/Sick Person,CBE,8600 block of Boteler Ln
12916,2026-00289167,2026-11-01 00:25:00,night,Fall 2026,Fall,2026-11-01 00:25:00,night,Fall 2026,Fall,Injured/Sick Person,CBE,3900 block of Campus Dr


## Merging detailed datasets

In [28]:
merged_dfs = pd.merge(dci_df, arrests_df, "left", on="umpd_case_number")

In [29]:
merged_dfs.to_csv(Path(RAW_DATA_DIR.parent / 'processed' / f"merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv"))

## Uniform Crime Reports

From the [UMPD's Uniform Crime Reports website](https://umpd.umd.edu/statistics-reports/uniform-crime-reports): 

> In 2021, the FBI retired the Summary Reporting System (SRS) and transitioned to a National Incident Based Reporting System (NIBRS) - a new national data collection standard. The University of Maryland Police Department (UMPD) began reporting using this new format in January of 2023...

So, year-to-year comparisons between before and after 2023 are hard to do. 

In [30]:
UCR_DIR = RAW_DATA_DIR / "uniform-crime-reports"

ucr_dfs = {}

for item in Path.iterdir(UCR_DIR):
  if item.is_file():
    ucr_dfs['srs_2012-2022'] = pd.read_csv(item)
  else:
    ucr_dfs[item.name] = {}
    for sub_item in Path.iterdir(item):
      ucr_dfs[item.name][sub_item.name[0:-13]] = pd.read_csv(sub_item).drop(['Unnamed: 0', 'Unnamed: 14'], axis=1)
    

In [31]:
for year in ucr_dfs:
  if year == "srs_2012-2022":
    continue
  ucr_dfs[year] = pd.concat(ucr_dfs[year]).reset_index().rename({'level_0': 'crime_type', year: 'crime'}, axis=1).drop('level_1', axis=1)
  ucr_dfs[year] = ucr_dfs[year].fillna(0.0)
  ucr_dfs[year]['year'] = year

/var/folders/l4/t5_22f1s0gg2brhv1z0457xw0000gn/T/ipykernel_22301/2527048809.py:5: Pandas4Warning: 'float' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  ucr_dfs[year] = ucr_dfs[year].fillna(0.0)
/var/folders/l4/t5_22f1s0gg2brhv1z0457xw0000gn/T/ipykernel_22301/2527048809.py:5: Pandas4Warning: 'float' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  ucr_dfs[year] = ucr_dfs[year].fillna(0.0)
/var/folders/l4/t5_22f1s0gg2brhv1z0457xw0000gn/T/ipykernel_22301/2527048809.py:5: Pandas4Warning: 'float' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  ucr_dfs[year] = ucr_dfs[year].fillna(0.0)
/var/folders/l4/t5_22f1s0gg2brh

In [32]:
valid_dfs = [df for year, df in ucr_dfs.items() if year != 'srs_2012-2022']
nibrs_df = pd.concat(valid_dfs)
nibrs_df = nibrs_df[(nibrs_df['crime'].str.endswith("Total") == False) & (nibrs_df['crime'].isna() == False) & (nibrs_df['crime'].isin(['0.0', 0.0, "", '0', 0]) == False)]
nibrs_df.head()

,crime_type,crime,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Total,year
0,crimes_against_property,Robbery,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,2025
1,crimes_against_property,Arson,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025
2,crimes_against_property,Extortion/Blackmail,0.0,0.0,1.0,1.0,2.0,0.0,0.0,0.0,1.0,3.0,3.0,2.0,13.0,2025
3,crimes_against_property,Burglary/Breaking or Entering,0.0,0.0,0.0,0.0,2.0,0.0,4.0,2.0,5.0,1.0,1.0,0.0,15.0,2025
4,crimes_against_property,Pocket-picking,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2025


In [33]:
nibrs_monthly = nibrs_df.melt(
  id_vars = ['year', 'crime', 'crime_type'],
  value_vars = MONTHS,
  var_name = 'month',
  value_name = 'count'
)
nibrs_monthly['count'] = nibrs_monthly['count'].astype('Int64')

In [34]:
ucr_path = RAW_DATA_DIR.parent / 'processed' / 'uniform-crime-reports'
if ucr_path.is_dir():
  shutil.rmtree(ucr_path.absolute())
  Path.mkdir(ucr_path)
else:
  Path.mkdir(ucr_path)

nibrs_df.to_csv(ucr_path / f"ucr_2023-{CURRENT_YEAR}.csv", index=False)
nibrs_monthly.to_csv(ucr_path / f"ucr_2023-{CURRENT_YEAR}_monthly.csv", index=False)